# SLM Inference: Reproduction Notebook

**Status:** Independently executed end-to-end in a fresh Google Colab Tesla T4 session (October 2026). This public notebook has its saved execution outputs cleared for readability. Use a GPU runtime; run cells in order. Performance may vary between sessions.

This notebook reproduces model-loading CUDA allocation deltas, generation benchmarks, and response-length checks for FP16 and 4-bit NF4. These measurements are not total GPU memory, peak memory, or energy usage.

The historical AI-assisted subjective quality scores are archived separately and are **not** regenerated or independently validated here; they are not human ratings.

Upload `data/benchmark_prompts.csv` to the Colab working directory before Section 3. Generated CSV outputs are written to that working directory. Original report results and separately labeled reproduction-run results should not be mixed.


## 1. Install dependencies and initialize

In [ ]:
%pip install -q "transformers==4.57.6" "accelerate==1.15.0" "bitsandbytes==0.50.2" "pandas==2.2.3" matplotlib

In [ ]:
import gc, time, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False)
print(torch.cuda.get_device_name(0))

## 2. Model-loading allocation measurement (historical procedure)

In [ ]:
import gc
import torch
import pandas as pd

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

memory_results = []

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def measure_memory(label, quant_config=None):
    cleanup()

    before = torch.cuda.memory_allocated()

    model_test = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="cuda:0",
        quantization_config=quant_config
    )

    model_test.eval()
    torch.cuda.synchronize()

    after = torch.cuda.memory_allocated()

    result = {
        "precision": label,
        "model_memory_gb": round(
            (after - before) / 1024**3, 4
        ),
        "allocated_after_load_gb": round(
            after / 1024**3, 4
        )
    }

    print(label, result)
    memory_results.append(result)

    del model_test
    cleanup()

measure_memory("FP16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

measure_memory("4-bit NF4", quant_config)

memory_df = pd.DataFrame(memory_results)
memory_df.to_csv("memory_benchmark.csv", index=False)

print("\n--- MEMORY COMPARISON ---")
print(memory_df.to_string(index=False))

## 3. Load the original 20-prompt benchmark set
Upload `data/benchmark_prompts.csv` into the Colab working directory before running this cell.

In [ ]:
from pathlib import Path
assert Path("benchmark_prompts.csv").exists(), "Upload data/benchmark_prompts.csv first"
prompt_df = pd.read_csv("benchmark_prompts.csv")
assert len(prompt_df) == 20 and prompt_df.prompt_id.is_unique
print("Loaded prompts:", len(prompt_df))

## 4. FP16 benchmark (40 timed generations)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda:0"
)

model_fp16.eval()

print("FP16 model ready.")
print("Device:", next(model_fp16.parameters()).device)

In [ ]:
import time
import pandas as pd
import torch

benchmark_results_fp16 = []

# Warm-up
warmup_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Hello"}],
    tokenize=False,
    add_generation_prompt=True
)

warmup_inputs = tokenizer(
    warmup_text,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    model_fp16.generate(
        **warmup_inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()

for _, row in prompt_df.iterrows():
    prompt_id = int(row["prompt_id"])
    prompt = row["prompt"]

    formatted = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to("cuda")

    for repeat in range(2):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model_fp16.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        input_length = inputs["input_ids"].shape[1]
        new_tokens = output.shape[1] - input_length

        answer = tokenizer.decode(
            output[0][input_length:],
            skip_special_tokens=True
        )

        benchmark_results_fp16.append({
            "precision": "FP16",
            "prompt_id": prompt_id,
            "repeat": repeat + 1,
            "input_tokens": input_length,
            "output_tokens": new_tokens,
            "latency_seconds": elapsed,
            "tokens_per_second": new_tokens / elapsed,
            "answer": answer
        })

    print(f"Completed prompt {prompt_id}/20")

df_fp16 = pd.DataFrame(benchmark_results_fp16)

df_fp16.to_csv(
    "multiprompt_fp16_results.csv",
    index=False
)

print("\nFP16 benchmark completed!")
print("Total runs:", len(df_fp16))
print(
    "Average throughput:",
    round(df_fp16["tokens_per_second"].mean(), 2),
    "tokens/sec"
)
print(
    "Average latency:",
    round(df_fp16["latency_seconds"].mean(), 3),
    "seconds"
)

In [ ]:
import gc
import torch

# Remove FP16 model
del model_fp16

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("FP16 model removed.")
print(
    "GPU allocated memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GiB"
)

## 5. 4-bit NF4 benchmark (40 timed generations)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    quantization_config=quant_config,
    device_map="cuda:0"
)

model_4bit.eval()

print("4-bit model ready.")
print("Device:", next(model_4bit.parameters()).device)

In [ ]:
import time
import pandas as pd
import torch

benchmark_results_4bit = []

# Warm-up
warmup_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Hello"}],
    tokenize=False,
    add_generation_prompt=True
)

warmup_inputs = tokenizer(
    warmup_text,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    model_4bit.generate(
        **warmup_inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()

for _, row in prompt_df.iterrows():
    prompt_id = int(row["prompt_id"])
    prompt = row["prompt"]

    formatted = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to("cuda")

    for repeat in range(2):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model_4bit.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        input_length = inputs["input_ids"].shape[1]
        new_tokens = output.shape[1] - input_length

        answer = tokenizer.decode(
            output[0][input_length:],
            skip_special_tokens=True
        )

        benchmark_results_4bit.append({
            "precision": "4-bit NF4",
            "prompt_id": prompt_id,
            "repeat": repeat + 1,
            "input_tokens": input_length,
            "output_tokens": new_tokens,
            "latency_seconds": elapsed,
            "tokens_per_second": (
                new_tokens / elapsed
            ),
            "answer": answer
        })

    print(f"Completed prompt {prompt_id}/20")

df_4bit = pd.DataFrame(benchmark_results_4bit)

df_4bit.to_csv(
    "multiprompt_4bit_results.csv",
    index=False
)

print("\n4-bit benchmark completed!")
print("Total runs:", len(df_4bit))
print(
    "Average throughput:",
    round(df_4bit["tokens_per_second"].mean(), 2),
    "tokens/sec"
)
print(
    "Average latency:",
    round(df_4bit["latency_seconds"].mean(), 3),
    "seconds"
)

## 6. Extended response generation (same base prompts, 100-word instruction)

In [ ]:
# Prepare the existing 20 public prompts
# for the extended quality evaluation

extended_quality_df = prompt_df[
    ["prompt_id", "prompt"]
].copy()

extended_quality_df["prompt"] = (
    extended_quality_df["prompt"]
    + " Answer clearly and concisely in no more than 100 words."
)

extended_quality_df.to_csv(
    "extended_quality_prompts.csv",
    index=False
)

print("Total evaluation prompts:", len(extended_quality_df))
print(
    extended_quality_df[
        ["prompt_id", "prompt"]
    ].head(5).to_string(index=False)
)

print("\nExtended quality prompts saved!")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM

# Free the previous model
if "model_4bit" in globals():
    del model_4bit

gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda:0"
)

model_fp16.eval()

print("FP16 model loaded successfully!")
print("Total prompts:", len(extended_quality_df))

In [ ]:
import pandas as pd
import torch

extended_fp16_results = []

for _, row in extended_quality_df.iterrows():

    prompt_id = int(row["prompt_id"])
    prompt = row["prompt"]

    formatted = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        output = model_fp16.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = output[0][input_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    output_tokens = len(generated_ids)
    word_count = len(answer.split())

    # Qwen may use im_end as its end-of-turn token
    last_token = generated_ids[-1].item()

    ended_normally = last_token in {
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>")
    }

    extended_fp16_results.append({
        "precision": "FP16",
        "prompt_id": prompt_id,
        "prompt": prompt,
        "answer": answer,
        "output_tokens": output_tokens,
        "word_count": word_count,
        "ended_normally": ended_normally,
        "within_100_words": word_count <= 100,
        "hit_token_limit": output_tokens >= 256
    })

    print(
        f"Prompt {prompt_id}/20 | "
        f"Words: {word_count} | "
        f"Tokens: {output_tokens} | "
        f"Ended: {ended_normally}"
    )

extended_fp16_df = pd.DataFrame(
    extended_fp16_results
)

extended_fp16_df.to_csv(
    "extended_quality_fp16.csv",
    index=False
)

print("\nFP16 extended quality evaluation completed!")
print("Total responses:", len(extended_fp16_df))
print(
    "Within 100 words:",
    extended_fp16_df["within_100_words"].sum()
)
print(
    "Ended normally:",
    extended_fp16_df["ended_normally"].sum()
)

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Free FP16 model
if "model_fp16" in globals():
    del model_fp16

gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float16,
    quantization_config=quant_config,
    device_map="cuda:0"
)

model_4bit.eval()

print("4-bit NF4 model loaded successfully!")
print("Total prompts:", len(extended_quality_df))

In [ ]:
import pandas as pd
import torch

extended_4bit_results = []

end_token_ids = {
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
}

for _, row in extended_quality_df.iterrows():

    prompt_id = int(row["prompt_id"])
    prompt = row["prompt"]

    formatted = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        output = model_4bit.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = output[0][input_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    output_tokens = len(generated_ids)
    word_count = len(answer.split())

    ended_normally = (
        generated_ids[-1].item() in end_token_ids
    )

    extended_4bit_results.append({
        "precision": "4-bit NF4",
        "prompt_id": prompt_id,
        "prompt": prompt,
        "answer": answer,
        "output_tokens": output_tokens,
        "word_count": word_count,
        "ended_normally": ended_normally,
        "within_100_words": word_count <= 100,
        "hit_token_limit": output_tokens >= 256
    })

    print(
        f"Prompt {prompt_id}/20 | "
        f"Words: {word_count} | "
        f"Tokens: {output_tokens} | "
        f"Ended: {ended_normally}"
    )

extended_4bit_df = pd.DataFrame(
    extended_4bit_results
)

extended_4bit_df.to_csv(
    "extended_quality_4bit.csv",
    index=False
)

print("\n4-bit extended quality evaluation completed!")
print("Total responses:", len(extended_4bit_df))
print(
    "Within 100 words:",
    extended_4bit_df["within_100_words"].sum()
)
print(
    "Ended normally:",
    extended_4bit_df["ended_normally"].sum()
)

In [ ]:
import pandas as pd

comparison = extended_fp16_df.merge(
    extended_4bit_df,
    on="prompt_id",
    suffixes=("_fp16", "_4bit"),
    validate="one_to_one"
)

summary = comparison[
    [
        "prompt_id",
        "word_count_fp16",
        "word_count_4bit",
        "within_100_words_fp16",
        "within_100_words_4bit",
        "ended_normally_fp16",
        "ended_normally_4bit"
    ]
].copy()

summary.to_csv(
    "extended_quality_comparison.csv",
    index=False
)

print(summary.to_string(index=False))

print("\nComparison saved!")

## 7. Exploratory quality scores: historical data only

The original notebook contained a two-stage anonymization workflow and AI-assisted subjective scoring. To avoid presenting these ratings as newly obtained or independently validated, this clean notebook does **not** rerun or manufacture quality judgments. See `results/quality_scored_all_40.csv`, `results/paired_quality_statistics.csv`, and `docs/QUALITY_EVALUATION.md`.